In [ ]:
### import all necessary libraries
import os
import pandas as pd
import numpy as np
import openai
import requests
import time
from tqdm import tqdm
from openai import OpenAI
tqdm.pandas()

### This file processes the categorized data and extract methods out using deepseek

In [ ]:
input_folder = "/Users/anonymous_user/Downloads/experiment_results/exp3_categorization_results/LLaMa/temp"
output_folder = "/Users/anonymous_user/Downloads/experiment_results/exp4_attribute_analysis_results/extracted_methods_raw_data/llama"

In [ ]:
deepseek_api_key = os.environ["DEEPSEEK_API_KEY"]
client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

In [ ]:
prompt1 = """
    We have two strings: content (entire file) and extracted_snippet_full (a segment). Analyze the extracted_snippet_full by mapping it back to content and identify all methods it covers or partially covers. For each identified method, output the full method, including:

    Method declaration.
    Any preceding docstring.
    Preceding comments (if applicable).
    Full method code.
    If extracted_snippet_full overlaps with multiple methods, include all such methods in full. If no methods are covered, output "nothing", if the result does not contain a proper method declaration, it is "nothing". Return only the methods or "nothing" with no additional text or explanation.
    """
def use_deepseek(client, prompt, content, extracted_snippet):
    response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": f"{prompt}"},
        {"role": "user", "content": f"content is {content}, extracted_snippet_full is {extracted_snippet}"},
    ],
    stream=False,
    temperature=0.0,
    )

    return response.choices[0].message.content
def process_row(row):
    return use_deepseek(
        client, 
        prompt1, 
        row["content"], 
        row["extracted_snippet_full"]
    )

In [ ]:
prompt2 = """ 
    The results string is a code snippet which may consists one or multiple or no methods. 
    Process the string, extract methods in this snippet, and return them to a list, each item in that list is a method, if there is no method, return an empty list.
    ONLY return methods with documentation (or comments) and declarion.
    Make sure to include the method declaration, docstring, comments, and code, if there is comment precedding the declaration, make sure to include that in the method as well.
    Don't say anything else, just output the result in this format: method1, method2, method3, ..., each method separated by 'xxxxx' if there is no method, return "nothing".
    """

def deepseek_method_extraction(client, prompt, results):
    
    response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": f"{prompt}"},
        {"role": "user", "content": f"process this {results}"},
    ],
    stream=False,
    temperature=0.0,
    )

    return response.choices[0].message.content


def method_extraction(row):
    return deepseek_method_extraction(
        client, 
        prompt2, 
        row["result_1st_step"]
    )

In [ ]:
prompt3 = """ 
    The input is python method, read and analyze it and process it into three different parts: declaration, docstring and/or comments, and code.
    Output the results separately, DO NOT say anything else, just output the result directly! Separate the three parts with 'xxx', the output is in the order of: delcaration xxx docstring and/or comments xxx code.
    """

def deepseek_method_analysis(client, prompt, results):
    
    response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": f"{prompt}"},
        {"role": "user", "content": f"Analyze: {results}"},
    ],
    stream=False,
    temperature=0.0,
    )

    return response.choices[0].message.content


def method_analysis(row):
    return deepseek_method_extraction(
        client, 
        prompt3, 
        row["all_methods"]
    )

In [ ]:
for file in os.listdir(input_folder):
    if file.endswith(".csv"):
        # Read the CSV file
        df = pd.read_csv(os.path.join(input_folder, file))

        df_code = df[df['category_better_prompt_deepseek'] == 'code']
        df_code['result_1st_step'] = df_code.progress_apply(process_row, axis = 1)
        df_code = df_code[df_code['result_1st_step'] != "nothing"]
        df_code['methods'] = df_code.progress_apply(method_extraction, axis = 1)
        df_code = df_code[df_code['methods'] != "nothing"]
        df_code['all_methods'] = df_code['methods'].str.split("xxxxx")
        df_code_final = df_code.explode('all_methods')
        df_code_final['method_parts'] = df_code_final.progress_apply(method_analysis, axis = 1)
        df_code_final[['declaration', 'comment', 'code']] = df_code_final['method_parts'].str.split("xxx", n=2, expand=True)
        file_path = os.path.join(output_folder, file)
        df_code_final.to_csv(file_path, index=False)
        print(f"Processed {file} and saved the results.")